# 45 — Export to LaTeX

Collects every figure and table that notebooks 40–44 saved into one folder for the thesis:

- `latex/figures/*.pdf` — vector figures, drawn at their printed width;
- `latex/tables/*.tex` — booktabs tables;
- `latex/figures.tex` — one `figure` block per figure (graphic, caption, label);
- `latex/tables.tex` — `\input` of every table;
- `latex/FIGURE_INDEX.md` — item → question → LaTeX label → file → source.

It computes nothing: it only reads what the question notebooks saved. Re-run it after re-running any of them.

In the thesis preamble: `\usepackage{graphicx,booktabs,amssymb}`.

In [1]:
import sys, pathlib, re
PILLAR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(PILLAR))
import pandas as pd
from viz import config, latex
cfg = config.load()

## Export

In [2]:
res = latex.export(cfg)
out = cfg.out('latex')
print(len(res['figures']), 'figures and', len(res['tables']), 'tables written to', out)

25 figures and 42 tables written to /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/latex


## Check the LaTeX

Every file must have balanced braces and matching `\begin`/`\end`; every figure block must point at a PDF that exists.

In [3]:
def balanced(s):
    s = re.sub(r'\\[{}]', '', s)
    depth = 0
    for ch in s:
        depth += (ch == '{') - (ch == '}')
        if depth < 0: return False
    return depth == 0
rows = []
for f in [out / 'figures.tex', out / 'tables.tex', *sorted((out / 'tables').glob('*.tex'))]:
    s = f.read_text(encoding='utf-8')
    envs = re.findall(r'\\(begin|end)\{(\w+)\}', s)
    ok_env = sum(1 for b, e in envs if b == 'begin') == sum(1 for b, e in envs if b == 'end')
    rows.append(dict(file=f.name, braces=balanced(s), environments=ok_env))
missing = [p for p in re.findall(r'figures/([\w.-]+\.pdf)', (out / 'figures.tex').read_text()) if not (out / 'figures' / p).exists()]
chk = pd.DataFrame(rows)
assert chk[['braces', 'environments']].all().all() and not missing, (chk, missing)
print('all files balanced; every figure PDF present')
chk

all files balanced; every figure PDF present


,file,braces,environments
0,figures.tex,True,True
1,tables.tex,True,True
2,fig_06_i_units_per_aircraft.tex,True,True
3,fig_06_ii_arrangement_share.tex,True,True
4,tab_10_cannot_answer.tex,True,True
5,tab_1_3_5_dataset_vs_directory.tex,True,True
6,tab_1_3_5_directory_subsets.tex,True,True
7,tab_1_ari_timeline.tex,True,True
8,tab_2_public_pairwise.tex,True,True
9,tab_3_trl_by_class.tex,True,True


## Index

In [4]:
print((out / 'FIGURE_INDEX.md').read_text(encoding='utf-8'))

# Figure and table index

Generated by notebook 45 from the outputs of notebooks 40–44.

| Item | Question | LaTeX label | File | Source |
|---|---|---|---|---|
| fig_01_filings_per_year | q1 | fig:fig-01-filings-per-year | figures/fig_01_filings_per_year.pdf | PatSeer (snapshot 2026-06-08) · human labelling · sector events: the author's dated list (checked 2026-09-23) · baselines: PatSeer worldwide simple families for CPC B64 and CPC section B, WIPO patent families by origin (indicator 6a) for all patents, exported 2026-09-25 |
| fig_02_evtolnews_check | q1 | fig:fig-02-evtolnews-check | figures/fig_02_evtolnews_check.pdf | PatSeer · human labelling, folded to the five classes of the directory (analysis.class_fold) · evtol.news directory (Vertical Flight Society), crawled 2026-09-17, development year read by two readers |
| fig_03_ari_clock | q1 | fig:fig-03-ari-clock | figures/fig_03_ari_clock.pdf | AAM Reality Index (May 2026 release; first_flight and eis as the index prints them) ·

## One example block, as it goes into the thesis

In [5]:
print(res['figures'][0] if res['figures'] else 'no figures yet')

\begin{figure}[htbp]
\centering
\includegraphics[width=6.30in]{figures/fig_01_filings_per_year.pdf}
\caption{(i) unique aircraft, counted once, by the priority year of their first patent (later patents on the same aircraft are not counted again), per year and applicant region, with the representative patents as a line and the dated sector events under the bars. (ii) how fast each series grows: eVTOL architecture patents, eVTOL unique aircraft and three worldwide baselines (WIPO), each relative to its own 2005-09 average on a log axis, with its fitted line over 2005-2023. (i) filings per priority year, by applicant region; (ii) eVTOL's share of aeronautics, of section B and of all patenting worldwide, relative to its 2005-09 share. \textit{Source:} PatSeer (snapshot 2026-06-08) $\cdot$ human labelling $\cdot$ sector events: the author's dated list (checked 2026-09-23) $\cdot$ baselines: PatSeer worldwide simple families for CPC B64 and CPC section B, WIPO patent families by origin (indi